<a href="https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmanan2728/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!git clone https://github.com/abdulmanan2728/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 399, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 399 (delta 170), reused 110 (delta 100), pack-reused 178 (from 2)
Receiving objects: 100% (399/399), 1.97 MiB | 10.33 MiB/s, done.
Resolving deltas: 100% (225/225), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter


In [8]:
import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

## 1. Build the feature vector

Engineered features from the starter CSV: numeric performance/engagement columns
used directly; `ctr_gap` engineered as CTR relative to position-tier peers (raw CTR
alone isn't comparable across position tiers).

In [9]:
feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'age_tier_order', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]

median_ctr_by_tier = df.groupby('position_tier')['ctr'].transform('median')
df['ctr_gap'] = median_ctr_by_tier - df['ctr']

missing_report = df[feature_cols].isnull().sum()
print("missing values per feature:")
print(missing_report[missing_report > 0])
print(f"\nfeature vector shape: {df[feature_cols].shape}")

missing values per feature:
search_volume    2468
competition      2468
cpc              2468
word_count       7699
char_count       7699
scroll_rate       125
dtype: int64

feature vector shape: (30000, 29)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 2. Feature notes

- **Volume features** (`impressions_90d`, `clicks_90d`, etc.): raw counts, no missing
  values in this dataset — available at decision time since they're historical.
- **`avg_position`**: 0 means "no data," per the data dictionary, not rank zero.
- **`ctr`**: a ×100 percentage (0.76 means 0.76%, not 76%).
- **Categorical fields** (`content_type`, `main_intent`, `freshness_tier`) not
  included as numeric features here — would need encoding, out of scope for this
  vector.
- All features exist BEFORE the label's outcome window — none depend on future data.

In [10]:
print(df['avg_position'].eq(0).sum(), "rows with avg_position == 0 (no data, not rank 0)")
print(df['ctr'].describe())

1205 rows with avg_position == 0 (no data, not rank 0)
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## 3. The leakage hunt

Attack: check every feature's correlation with `trend_pct` (the label's source
signal), and confirm no label-derived column made it into the feature list.

In [11]:
leak_corr = df[feature_cols].corrwith(df['trend_pct']).abs().sort_values(ascending=False)
print(leak_corr.head(5))

label_cols = ['trend_pct', 'trend_direction', 'is_declining']
overlap = set(feature_cols) & set(label_cols)
print(f"\nlabel-derived columns in feature set: {overlap}")  # should be empty set

impressions_last_30d     0.096737
avg_position             0.047248
days_with_impressions    0.031776
impressions_90d          0.024187
competition              0.014426
dtype: float64

label-derived columns in feature set: set()


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I excluded and why

- `trend_pct`, `trend_direction` — these ARE the label's source; using them as
  features would let the model learn the label definition, not a real pattern.
- `content_id`, `client_id` — pseudonymous IDs, used only for grouping/splitting,
  never as predictive features.
- `provider_used`, `model_used` — describe content production method, not search
  performance; out of scope for a decline-detection signal.

In [12]:
print("No query needed — this section states exclusions and reasoning.")

No query needed — this section states exclusions and reasoning.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.